In [1]:
# downloading dataset
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data =response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8")as file:
            file.write(text_data)
    else:
        with open(file_path, "r",encoding="utf-8") as file:
            text_data =file.read()

    with open(file_path, "r") as file:
        data =json.load(file)
    return data

file_path ="instruction-data.json"
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
"/main/ch07/01_main-chapter-code/instruction-data.json"
)
data =download_and_load_file(file_path, url)
print("No. of entries: ", len(data))
print("Example entry: \n", data[567])


No. of entries:  1100
Example entry: 
 {'instruction': 'Identify the adjective in the sentence.', 'input': 'The red car sped down the road.', 'output': "The adjective in the sentence is 'red'."}


In [2]:
def format_input(entry):
    instruction_text= (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text =(
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text +input_text

In [3]:
# testing
model_input =format_input(data[567])
desired_response =f"\n\n### Response:\n{data[567]['output']}"
print(model_input +desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the adjective in the sentence.

### Input:
The red car sped down the road.

### Response:
The adjective in the sentence is 'red'.


In [4]:
# partioning dataste
train_portion = int(len(data)* 0.85)
test_portion =int(len(data)* 0.1)
val_portion =len(data) -train_portion -test_portion

train_data =data[:train_portion]
test_data = data[train_portion :train_portion+ test_portion]
val_data =data[train_portion +test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


In [5]:
# instruction dataset class
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data,tokenizer):
        self.data =data
        self.encoded_texts =[]

        # pretokenizing text
        for entry in data:
            instruction_plus_input =format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text =instruction_plus_input +response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]
    def __len__(self):
        return len(self.data)

In [6]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [ ]:
# pads training examples in each batch to the same length while allowing different batches to have diff lengths
def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):
    # longest sequence in batch
    batch_max_length =max(len(item) +1 for item in batch)   
    inputs_lst = []

    for item in batch:
        # padds and preparing inputs
        new_item=item.copy()
        new_item +=[pad_token_id]
        padded =(new_item +[pad_token_id]*(batch_max_length -len(new_item)))
        inputs =torch.tensor(padded[: -1])
        inputs_lst.append(inputs)

    inputs_tensor =torch.stack(inputs_lst).to(device)
    return inputs_tensor

In [8]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (inputs_1,inputs_2,inputs_3)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


In [9]:
# generates target token ids from input token ids 
def custom_collate_draft_2(batch, pad_token_id=50256, device="cpu"):
    batch_max_length =max(len(item) +1 for item in batch)
    inputs_lst, targets_lst= [],[]

    for item in batch:
        new_item =item.copy()
        new_item +=[pad_token_id]
        padded =(new_item +[pad_token_id]*(batch_max_length -len(new_item)))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        inputs_lst.append(inputs)
        targets_lst.append(targets)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

inputs, targets =custom_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


In [11]:
# custom batch collate function
def custom_collate_fn(batch, pad_token_id=50256,ignore_index=-100, allowed_max_length=None,device="cpu"):
    batch_max_length =max(len(item) +1 for item in batch)
    inputs_lst,targets_lst =[],[]

    for item in batch:
        new_item =item.copy()
        new_item+= [pad_token_id]
        # pads sequences to max length
        padded= (new_item +[pad_token_id]*(batch_max_length -len(new_item)))

        # truncates last token for inputs
        inputs= torch.tensor(padded[: -1])
        targets= torch.tensor(padded[1:])

        # replaces all but first padding tokens in targets by ignore_index
        mask =targets ==pad_token_id
        indices =torch.nonzero(mask).squeeze()
        if indices.numel() >1:
            targets[indices[1: ]]= ignore_index

        if allowed_max_length is not None:
            inputs =inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor =torch.stack(inputs_lst).to(device)
    targets_tensor =torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [13]:
inputs, targets=custom_collate_fn(batch)
print("Inputs: ",inputs)
print("Targets:",targets)

Inputs:  tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
Targets: tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [15]:
# ex     #prediction for 1st   & 2nd token
logits_1 =torch.tensor(([-1, 1], [-0.5, 1.5]))
targets_1 = torch.tensor([0,1])
loss_1 =torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

logits_2 = torch.tensor(
[[-1.0, 1.0],
[-0.5, 1.5],
[-0.5, 1.5]]
)
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(1.1269)
tensor(0.7936)


In [17]:
# replacing 3rd target token id with -100
targets_3 =torch.tensor([0, 1,-100])
loss_3 =torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3: ",loss_1 == loss_3)

tensor(1.1269)
loss_1 == loss_3:  tensor(True)
